## Setup

In [4]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [26]:
from pathlib import Path
import networkx as nx
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import contextily as ctx
import xyzservices as xyz
import matplotlib.pyplot as plt

# import kml reading and set supported driver
import fiona

fiona.drvsupport.supported_drivers["KML"] = "rw"

# for parsing HTML inside the Description field
from bs4 import BeautifulSoup

# for TIFF files
import rasterio
from rasterio.plot import show
from rasterstats import zonal_stats
from rasterio.features import geometry_mask
from rasterio import Affine
from rasterio.enums import Resampling

# for roads
import osmnx as ox

In [6]:
from gridsample.utils import create_ids, save_shapefiles
from gridsample.mapping.plot import create_interactive_map

In [7]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
PROCESSED_DATA_DIR = DATA_DIR / "01_processed" / "Solar Parks"

## Load processed shapes

In [ ]:
dhar_processed_areas_gdf = gpd.read_parquet(PROCESSED_DATA_DIR / "processed_areas" / "dhar_processed_all.parquet")
dhar_processed_areas_gdf["source"] = "dhar"

In [ ]:
dhar_processed_areas_gdf["parcel_id"] = "DHAR_" + dhar_processed_areas_gdf["Name"]

# Function to add suffix to duplicates
def add_suffix_to_duplicates(series):
    counts = {}
    result = []
    for item in series:
        if item in counts:
            counts[item] += 1
            result.append(f"{item}_{counts[item]}")
        else:
            counts[item] = 0
            result.append(item)
    return result

# Apply the function to the parcel_id column
dhar_processed_areas_gdf["parcel_id"] = add_suffix_to_duplicates(dhar_processed_areas_gdf["parcel_id"])

In [10]:
sagar_gdf = gpd.read_parquet(PROCESSED_DATA_DIR / "processed_areas" / "sagar_processed_all.parquet")
sagar_gdf["village_name"] = sagar_gdf["source"].str.split("_").str[1]
sagar_gdf["source"] = sagar_gdf["source"].str.split("_").str[0]

In [11]:
sagar_gdf["parcel_id"] = "SAGAR_" + sagar_gdf["UNQID"]

In [12]:
# filter to only the "PA" PAR_TYPE (since it looks like the barren land)
sagar_gdf = sagar_gdf[sagar_gdf["PAR_TYPE"] == "PA"]

In [13]:
gdf = pd.concat([dhar_processed_areas_gdf, sagar_gdf], ignore_index=True)

In [ ]:
gdf.plot()

In [15]:
gdf = gdf[["source", "village_name", "parcel_id", "UNQID", "Name", "geometry"]]

In [16]:
gdf = gdf.rename(columns={"village_name":"Source File Village Name", "UNQID": "SAGAR_UNQID", "Name": "DHAR_Name"})
gdf["Source File Village Name"].fillna("Not Available", inplace=True)

In [17]:
gdf["Parcel Area (m2)"] = gdf["geometry"].to_crs("24378").area.round(1)

In [18]:
gdf_stripped = gdf[["parcel_id", "geometry"]].copy()

## Combine khasras into continuous parcels

In [63]:
def calculate_pairwise_distances(gdf):
    distances = gdf.geometry.apply(lambda geom1: gdf.geometry.distance(geom1))
    return distances.to_numpy()

In [64]:
def cluster_adjacent_shapes(gdf, distance_threshold):

    gdf_index = gdf.index

    # Step 2: Create a spatial graph
    G = nx.Graph()
    for i, geom1 in enumerate(gdf.geometry):
        for j, geom2 in enumerate(gdf.geometry):
            if i != j and geom1.distance(geom2) < distance_threshold:
                G.add_edge(i, j)

    # Step 3: Find connected components
    connected_components = list(nx.connected_components(G))

    # Step 4: Convert the connected components to a list of labels that matches the input data
    data = []
    for cluster_id, value_set in enumerate(connected_components):
        for value in value_set:
            data.append((value, cluster_id))

    df = pd.DataFrame(data, columns=['index', 'cluster'])
    df.set_index('index', inplace=True)

    # make index of df the same as gdf and assign any missing values as -1
    cluster_labels = df.reindex(gdf_index)["cluster"].fillna(-1).astype(int)

    return cluster_labels

In [66]:
gdf = sagar_gdf.to_crs("24378").reset_index(drop=True)

In [67]:
dist_matrix = calculate_pairwise_distances(gdf)

In [68]:
gdf["cluster_50m"] = cluster_adjacent_shapes(gdf, distance_threshold=50)

In [ ]:
gdf.plot(column='cluster_50m', legend=True)

In [ ]:
# group by the cluster value and dissolve to make a new GeoDataFrame
clustered_gdf = gdf.dissolve(by='cluster_50m')
clustered_gdf["Area in hectares"] = clustered_gdf["geometry"].area / 10**4

In [ ]:
clustered_gdf[clustered_gdf["Area in hectares"] > 100].plot(column="Area in hectares", legend=True)

## Save files

In [162]:
# # Sagar only
# sagar_final = final_gdf[final_gdf["source"] == "sagar"]
# sagar_final = sagar_final.drop(columns=["DHAR_Name"])

# save_shapefiles(
#     sagar_final.to_crs(epsg=4326),
#     export_folderpath,
#     "sagar_land_parcels_with_stats",
#     formats=["parquet", "kml", "csv"],
# )

## Scraps

In [ ]:
# sagar_gdf[sagar_gdf["UNQID"] == "17677985"].geometry.values[0].distance(
#     sagar_gdf[sagar_gdf["UNQID"] == "17677984"].geometry.values[0]
# )

In [ ]:
# sagar_gdf_4326 = sagar_gdf.to_crs("4326")
# sagar_gdf_4326["Lat"] = sagar_gdf_4326.centroid.y
# sagar_gdf_4326["Lon"] = sagar_gdf_4326.centroid.x
# create_interactive_map(sagar_gdf_4326, point_id_col="UNQID", zoom_start=12)

In [ ]:
# from sklearn.cluster import HDBSCAN
# clusters = HDBSCAN(min_cluster_size=2, metric="precomputed", n_jobs=-1).fit(dist_matrix)
# gdf['clustering_dbscan'] = clusters.labels_
# gdf.plot(column='clustering_dbscan', legend=True)